<a href="https://colab.research.google.com/github/vigneshavmm/TIH/blob/main/simple_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets torch accelerate peft bitsandbytes sentencepiece einops gguf mistral-common tokenizers
!pip install -q git+https://github.com/huggingface/peft
!apt-get update -q && apt-get install -q build-essential


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 84.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease

In [4]:
files = ['/content/data.txt', '/content/data2.txt']
with open('merged.txt', 'w') as outfile:
    for fname in files:
        try:
            with open(fname, 'r', encoding='utf-8') as infile:
                outfile.write(infile.read() + "\n\n")
        except FileNotFoundError:
            print(f"[!] {fname} not found")
    print("text merged")


text merged


In [5]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
dataset = load_dataset('text', data_files='merged.txt')

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, num_proc=2)

Generating train split: 0 examples [00:00, ? examples/s]

Map (num_proc=2):   0%|          | 0/284 [00:00<?, ? examples/s]

In [8]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="gpt2-shrek-chaos",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    save_steps=100,
    logging_steps=50,
    learning_rate=3e-4,
    weight_decay=0.01,
    fp16=True,
    logging_dir="logs",
    report_to="none",
    save_total_limit=1,
    disable_tqdm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

print("training..")
trainer.train()

model.save_pretrained("gpt2-shrek-chaos")
tokenizer.save_pretrained("gpt2-shrek-chaos")
print("saved to 'gpt2-shrek-chaos'")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


training..


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to 'gpt2-shrek-chaos'


In [9]:
import torch
import torch.nn as nn

model = GPT2LMHeadModel.from_pretrained("gpt2-shrek-chaos")
model.eval()

weights_to_sparse = 0.5
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        with torch.no_grad():
            w = module.weight.data
            mask = torch.rand(w.shape) > weights_to_sparse
            w *= mask.to(w.device)

print(f"Model weights sparsified ({weights_to_sparse} zeroed out)")
model.save_pretrained("gpt2-shrek-chaos-sparse")
tokenizer.save_pretrained("gpt2-shrek-chaos-sparse")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model weights sparsified (0.5 zeroed out)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('gpt2-shrek-chaos-sparse/tokenizer_config.json',
 'gpt2-shrek-chaos-sparse/tokenizer.json')

In [10]:
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release && make -j$(nproc)

Cloning into 'llama.cpp'...
remote: Enumerating objects: 92943, done.
remote: Total 92943 (delta 0), reused 0 (delta 0), pack-reused 92943 (from 1)
Receiving objects: 100% (92943/92943), 387.30 MiB | 20.62 MiB/s, done.
Resolving deltas: 100% (66074/66074), done.
Updating files: 100% (2746/2746), done.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREA

In [11]:
!cp -r gpt2-shrek-chaos-sparse llama.cpp/models/gpt2-shrek-chaos-sparse
!cd llama.cpp && python3 convert_hf_to_gguf.py models/gpt2-shrek-chaos-sparse --outfile models/gpt2-shrek-chaos.gguf

INFO:hf-to-gguf:Loading model: gpt2-shrek-chaos-sparse
INFO:hf-to-gguf:Model architecture: GPT2LMHeadModel
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics unable to detect tensor dtype, defaulting to --outtype f16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:blk.0.attn_qkv.bias,       torch.float32 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.attn_qkv.weight,     torch.float32 --> F16, shape = {768, 2304}
INFO:hf-to-gguf:blk.0.attn_output.bias,    torch.float32 --> F32, shape = {768}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float32 --> F16, shape = {768, 768}
INFO:hf-to-gguf:blk.0.attn_norm.bias,      torch.float32 --> F32, shape = {768}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float32 --> F32, shape = {768}
INFO:hf-to-gguf:blk.0.ffn_norm.bias,       torch.float32 --> F32, shape = {768}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float32 --> F32, sha

In [12]:
!cd llama.cpp && ./build/bin/llama-quantize models/gpt2-shrek-chaos.gguf models/gpt2-shrek-chaos-Q2_K.gguf Q2_K
!ls -lh llama.cpp/models/*.gguf

llama_print_build_info: build = 9128 (856c3adac)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'models/gpt2-shrek-chaos.gguf' to 'models/gpt2-shrek-chaos-Q2_K.gguf' as Q2_K
llama_model_loader: loaded meta data with 21 key-value pairs and 148 tensors from models/gpt2-shrek-chaos.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gpt2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Gpt2 Shrek Chaos Sparse
llama_model_loader: - kv   3:                         general.size_label str              = 124M
llama_model_loader: - kv   4:                           gpt2.block_count u32              = 12
llama_model_loader: - kv   5:                        g

In [13]:
!cd llama.cpp && ./build/bin/llama-cli \
    -m models/gpt2-shrek-chaos-Q2_K.gguf \
    -p "1+1 = " \
    -n 100 \
    --temp 1.0 \
    --no-warmup \
    --repeat-penalty 2.0


Loading model... |-\|/-\|/-\|/-\|/ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b9128-856c3adac
model      : gpt2-shrek-chaos-Q2_K.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern


> 1+1 = 

|-\|/-\ optional=AF ready receipts receipts receiptsac BayTheat MobUnit unsu dispatcherthispercent.<//content/llama.cpp/build/bin/libggml-base.so.0(+0x1b27b)[0x7e0f17a0927b]
/content/llama.cpp/build/bin/libggml-base.so.0(ggml_print_backtrace+0x21f)[0x7e0f17a096ff]
/content/llama.cpp/build/bin/libggml-base.so.0(+0x2f51f)[0x7e0f17a1d51f]
/

In [14]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

tokenizer = GPT2Tokenizer.from_pretrained("gpt2-shrek-chaos")
model = GPT2LMHeadModel.from_pretrained("gpt2-shrek-chaos")
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Chat with Shrek GPT2. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break

    inputs = tokenizer(user_input, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Bot: {response}\n")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Chat with Shrek GPT2. Type 'quit' to exit.

You: hi
Bot: hi and raster masking algorithms combine precision detection values between pixels based on spectral bands. They are a high-performance, small computing environment for large data sets with no real training time or validation analysis cost constraints in place. OpenMP is open source that supports crop mapping using the EPSGIR pipeline used across clouds as input layer classification points from Cloud NAC to Point classifier labels where suitable correction errors may be detected before calibration due use of cloud masks over long distances prior accuracy measurements have been collected at

You: tirupati terrian layer analysis
Bot: tirupati terrian layer analysis uses STIFs of 128-192 polygons to detect pixels in space. PostGIS autocorrelation is precision between samples, with minimum negative values representing a statistically significant difference before correction and maximum positives represent an edge loss or gain event w